In [1]:
# ===== Cell 1: Settings, load, checks =====
# Notebook: P3_M_Hazard_Model_v03_final.ipynb
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, pairwise_distances
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
pd.set_option("display.width", 220)

ROOT = Path(r"D:\Siam\Thesis_GeoAI")
P_MASTER = ROOT / "04_master_table" / "master_v06_20261020.csv"
P_LABEL = ROOT / "03_features" / "Label_Flooded_Event_v02.csv"
P_MASK = ROOT / "03_features" / "Observability_Mask_v01.csv"
P_GRID = ROOT / "02_boundary_grid" / "grid_250m_v01.gpkg"
P_HAZ_V02 = ROOT / "05_models" / "final_v02" / "hazard_predictions_v02.csv"
OUT = ROOT / "05_models" / "final_v03"
OUT.mkdir(parents=True, exist_ok=True)

FEATURES = ["Elevation_Mean_m", "Slope_Mean_deg", "Distance_to_River_m", "Cropland_Fraction"]   # FROZEN
TARGET = "Flooded_Event"
BLOCK_M = 15000
N_FOLDS = 5
N_BOOT = 500
FR_BINS = 10
TREE_TRANSFER_TRAIN_MAX = 0.1
TREE_TRANSFER_TEST = (0.1, 0.3)
RF_PARAMS = dict(n_estimators=500, min_samples_leaf=20, max_features=2, n_jobs=-1, random_state=SEED)
LGBM_PARAMS = dict(n_estimators=500, learning_rate=0.03, num_leaves=31, min_child_samples=50,
                   subsample=0.8, subsample_freq=1, random_state=SEED, n_jobs=-1, verbose=-1)
MODELS = ["ElevOnly", "FR", "Logistic", "RF", "LGBM"]
N_MODEL = 160874
N_LABEL_ROWS = 457251

m = pd.read_csv(P_MASTER, usecols=list(dict.fromkeys(
    ["Grid_ID", "Is_Water", "Tree_Fraction", "Mangrove_Fraction_GMW", "Population_Count", "District"] + FEATURES)))
cells = m.loc[m["Is_Water"] == 0].reset_index(drop=True)
assert len(cells) == N_MODEL and not cells["Grid_ID"].duplicated().any()
assert cells[FEATURES].notna().all().all(), "NaN in features"
assert (cells["Slope_Mean_deg"] >= 0).all() and (cells["Distance_to_River_m"] >= 0).all()
assert cells["Cropland_Fraction"].between(0, 1).all()
print("Feature ranges (model cells):")
print(cells[FEATURES].describe().T[["min", "50%", "max"]].round(3).to_string())

mask = pd.read_csv(P_MASK)
cells = cells.merge(mask[["Grid_ID", "Is_Observable", "Unobservable_Reason", "Has_Label_Row"]],
                    on="Grid_ID", how="left", validate="1:1")
assert cells["Is_Observable"].notna().all()

grid = gpd.read_file(P_GRID)
assert grid.crs.to_epsg() == 32646, f"Grid CRS {grid.crs}"
cent = grid.geometry.centroid
xy = pd.DataFrame({"Grid_ID": grid["Grid_ID"].values, "X_m": cent.x.values, "Y_m": cent.y.values})
if pd.api.types.is_numeric_dtype(xy["Grid_ID"]) and pd.api.types.is_numeric_dtype(cells["Grid_ID"]):
    xy["Grid_ID"] = xy["Grid_ID"].astype("int64")
    cells["Grid_ID"] = cells["Grid_ID"].astype("int64")
cells = cells.merge(xy, on="Grid_ID", how="left", validate="1:1")
assert cells["X_m"].notna().all()
cells["Block_ID"] = (np.floor(cells["X_m"] / BLOCK_M).astype(int).astype(str) + "_" +
                     np.floor(cells["Y_m"] / BLOCK_M).astype(int).astype(str))

lab = pd.read_csv(P_LABEL)
assert len(lab) == N_LABEL_ROWS
assert ((lab["Is_Observable"] == 0) == lab[TARGET].isna()).all(), "NaN labels do not match mask"
tr = (lab.loc[lab[TARGET].notna()].drop(columns=["District", "Is_Observable"], errors="ignore")
      .merge(cells[["Grid_ID", "Block_ID", "Tree_Fraction", "District"] + FEATURES],
             on="Grid_ID", how="left", validate="m:1"))
tr = tr.reset_index(drop=True)
tr["y"] = tr[TARGET].astype(int)
assert tr[FEATURES].notna().all().all()
EVENTS = sorted(tr["Event_ID"].unique())
print(f"\nTraining rows: {len(tr):,} | cells: {tr['Grid_ID'].nunique():,} | blocks: {tr['Block_ID'].nunique()}")
print("Base rate by event (%):", (100 * tr.groupby("Event_ID")["y"].mean()).round(3).to_dict())
print("Max Tree_Fraction in training:", round(float(tr["Tree_Fraction"].max()), 3))
print("\nCell 1 OK")

Feature ranges (model cells):
                       min      50%        max
Elevation_Mean_m    -0.421    2.267     25.148
Slope_Mean_deg       0.000    0.164      9.430
Distance_to_River_m  0.000  987.885  17054.236
Cropland_Fraction    0.000    0.394      1.000

Training rows: 148,719 | cells: 49,573 | blocks: 73
Base rate by event (%): {'Amphan': 7.908, 'Remal': 27.507, 'Yaas': 4.692}
Max Tree_Fraction in training: 0.3

Cell 1 OK


In [2]:
# ===== Cell 2: Model helpers + spatial folds =====
# Notebook: P3_M_Hazard_Model_v03_final.ipynb
def fr_fit(X, y):
    tables, n_pos = {}, y.sum()
    for c in X.columns:
        inner = np.unique(np.quantile(X[c], np.linspace(0, 1, FR_BINS + 1)[1:-1]))
        b = np.searchsorted(inner, X[c].values, side="right")
        nb = len(inner) + 1
        n = np.bincount(b, minlength=nb)
        p = np.bincount(b, weights=y, minlength=nb)
        tables[c] = (inner, (p / n_pos) / np.maximum(n / len(y), 1e-12))
    return tables


def fr_predict(tables, X):
    s = np.zeros(len(X))
    for c, (inner, fr) in tables.items():
        s += fr[np.searchsorted(inner, X[c].values, side="right")]
    return s


def fit_model(name, X, y):
    if name == "ElevOnly":
        return None
    if name == "FR":
        return fr_fit(X, y)
    if name == "Logistic":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X, y)
    if name == "RF":
        return RandomForestClassifier(**RF_PARAMS).fit(X, y)
    if name == "LGBM":
        return lgb.LGBMClassifier(**LGBM_PARAMS).fit(X, y)
    raise ValueError(name)


def predict_model(name, model, X):
    if name == "ElevOnly":
        return -X["Elevation_Mean_m"].values
    if name == "FR":
        return fr_predict(model, X)
    return model.predict_proba(X)[:, 1]


def scores(y, s):
    pr = average_precision_score(y, s)
    return {"N": len(y), "Base_Rate": y.mean(), "ROC_AUC": roc_auc_score(y, s),
            "PR_AUC": pr, "PR_Lift": pr / y.mean()}


sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
tr["Fold"] = -1
for k, (_, te) in enumerate(sgkf.split(tr[FEATURES], tr["y"], groups=tr["Block_ID"])):
    tr.loc[te, "Fold"] = k
assert (tr["Fold"] >= 0).all()
assert tr.groupby("Grid_ID")["Fold"].nunique().max() == 1, "A cell is split across folds"
assert tr.groupby("Block_ID")["Fold"].nunique().max() == 1, "A block is split across folds"
print(tr.groupby("Fold").agg(Rows=("y", "size"), Positives=("y", "sum"), Blocks=("Block_ID", "nunique")).to_string())
print("Cell 2 OK")

       Rows  Positives  Blocks
Fold                          
0     31854       4078      14
1     43110       4258      18
2     17814       2317      13
3     25617       5192      11
4     30324       4037      17
Cell 2 OK


In [3]:
# ===== Cell 3: Spatial block CV (OOF) + calibration =====
# Notebook: P3_M_Hazard_Model_v03_final.ipynb
t0 = time.time()
oof = pd.DataFrame(index=tr.index, columns=MODELS, dtype=float)
for k in range(N_FOLDS):
    trn = (tr["Fold"] != k).values
    Xtr, ytr, Xte = tr.loc[trn, FEATURES], tr.loc[trn, "y"].values, tr.loc[~trn, FEATURES]
    for name in MODELS:
        oof.loc[~trn, name] = predict_model(name, fit_model(name, Xtr, ytr), Xte)
    print(f"fold {k} done, {time.time() - t0:.0f} s")
assert oof.notna().all().all()

rows = []
for name in MODELS:
    rows.append({"Model": name, "Event": "Pooled", **scores(tr["y"].values, oof[name].values)})
    for e in EVENTS:
        i = (tr["Event_ID"] == e).values
        rows.append({"Model": name, "Event": e, **scores(tr["y"].values[i], oof[name].values[i])})
met = pd.DataFrame(rows).round(4)
met.to_csv(OUT / "metrics_v03.csv", index=False)
pd.concat([tr[["Grid_ID", "Event_ID", "Block_ID", "Fold", "y"]], oof], axis=1).to_csv(OUT / "oof_v03.csv", index=False)
print(met.pivot(index="Model", columns="Event", values=["ROC_AUC", "PR_AUC", "PR_Lift"]).round(3).to_string())

for name in ["LGBM", "FR"]:
    top = oof[name].rank(pct=True) >= 0.9
    print(f"{name} top 10% rows capture {100 * tr.loc[top, 'y'].sum() / tr['y'].sum():.1f}% of positives")

# Calibration (probability models only)
yv = tr["y"].values
cal_rows, rel_rows = [], []
for name in ["Logistic", "RF", "LGBM"]:
    p = oof[name].values
    cal_rows.append({"Model": name, "Brier": brier_score_loss(yv, p),
                     "Brier_Climatology": brier_score_loss(yv, np.full(len(yv), yv.mean())),
                     "Mean_Pred": p.mean(), "Obs_Rate": yv.mean()})
    b = pd.qcut(p, 10, duplicates="drop")
    g = pd.DataFrame({"p": p, "y": yv}).groupby(b, observed=False).agg(Mean_Pred=("p", "mean"), Obs_Rate=("y", "mean"), N=("y", "size"))
    g["Model"] = name
    rel_rows.append(g.reset_index(drop=True))
cal = pd.DataFrame(cal_rows)
cal["Brier_Skill"] = 1 - cal["Brier"] / cal["Brier_Climatology"]
cal.round(4).to_csv(OUT / "calibration_v03.csv", index=False)
pd.concat(rel_rows).round(4).to_csv(OUT / "reliability_v03.csv", index=False)
print("\nCalibration:\n", cal.round(4).to_string(index=False))

fold 0 done, 70 s
fold 1 done, 135 s
fold 2 done, 215 s
fold 3 done, 293 s
fold 4 done, 367 s
         ROC_AUC                      PR_AUC                      PR_Lift                     
Event     Amphan Pooled  Remal   Yaas Amphan Pooled  Remal   Yaas  Amphan Pooled  Remal   Yaas
Model                                                                                         
ElevOnly   0.603  0.571  0.554  0.641  0.102  0.144  0.276  0.061   1.285  1.076  1.003  1.299
FR         0.628  0.697  0.738  0.761  0.125  0.253  0.517  0.126   1.578  1.896  1.880  2.693
LGBM       0.647  0.702  0.744  0.745  0.136  0.255  0.513  0.127   1.713  1.910  1.865  2.700
Logistic   0.613  0.666  0.698  0.722  0.109  0.201  0.404  0.096   1.383  1.506  1.470  2.058
RF         0.644  0.695  0.734  0.736  0.126  0.248  0.502  0.123   1.594  1.853  1.826  2.614
LGBM top 10% rows capture 22.9% of positives
FR top 10% rows capture 23.9% of positives

Calibration:
    Model  Brier  Brier_Climatology  Mean_Pr

In [4]:
# ===== Cell 4: Block bootstrap CIs =====
# Notebook: P3_M_Hazard_Model_v03_final.ipynb
t0 = time.time()
idx = tr.groupby("Block_ID").indices
blocks = np.array(list(idx.keys()))
y = tr["y"].values
S = {n: oof[n].values for n in MODELS}
rec = []
for i in range(N_BOOT):
    sel = rng.choice(blocks, size=len(blocks), replace=True)
    ii = np.concatenate([idx[b] for b in sel])
    yy = y[ii]
    if yy.sum() == 0 or yy.sum() == len(yy):
        continue
    r = {f"PR_{n}": average_precision_score(yy, S[n][ii]) for n in MODELS}
    r.update({f"ROC_{n}": roc_auc_score(yy, S[n][ii]) for n in ["ElevOnly", "FR", "RF", "LGBM"]})
    rec.append(r)
    if i == 9:
        print(f"~{(time.time() - t0) / 10 * N_BOOT / 60:.1f} min estimated")
bt = pd.DataFrame(rec)
for a, b in [("RF", "FR"), ("LGBM", "FR"), ("LGBM", "RF"), ("LGBM", "ElevOnly"), ("FR", "ElevOnly")]:
    for mtr in ["PR", "ROC"]:
        if f"{mtr}_{a}" in bt and f"{mtr}_{b}" in bt:
            bt[f"{mtr}_{a}_minus_{b}"] = bt[f"{mtr}_{a}"] - bt[f"{mtr}_{b}"]
ci = bt.describe(percentiles=[0.025, 0.975]).T[["mean", "2.5%", "97.5%"]].round(4)
ci.to_csv(OUT / "bootstrap_ci_v03.csv")
print(f"Valid replicates: {len(bt)}")
print(ci.to_string())

~10.1 min estimated
Valid replicates: 500
                           mean    2.5%   97.5%
PR_ElevOnly              0.1457  0.1015  0.1955
PR_FR                    0.2531  0.1842  0.3138
PR_Logistic              0.2021  0.1473  0.2666
PR_RF                    0.2473  0.1811  0.3093
PR_LGBM                  0.2547  0.1872  0.3173
ROC_ElevOnly             0.5708  0.5212  0.6244
ROC_FR                   0.6961  0.6607  0.7267
ROC_RF                   0.6943  0.6667  0.7210
ROC_LGBM                 0.7014  0.6726  0.7298
PR_RF_minus_FR          -0.0058 -0.0208  0.0087
ROC_RF_minus_FR         -0.0019 -0.0166  0.0162
PR_LGBM_minus_FR         0.0017 -0.0145  0.0178
ROC_LGBM_minus_FR        0.0052 -0.0099  0.0211
PR_LGBM_minus_RF         0.0074  0.0030  0.0124
ROC_LGBM_minus_RF        0.0071  0.0024  0.0120
PR_LGBM_minus_ElevOnly   0.1090  0.0684  0.1489
ROC_LGBM_minus_ElevOnly  0.1306  0.0787  0.1854
PR_FR_minus_ElevOnly     0.1073  0.0690  0.1464
ROC_FR_minus_ElevOnly    0.1253  0.0763  0.178

In [5]:
# ===== Cell 5: Leave-one-event-out (event + spatial block held out) =====
# Notebook: P3_M_Hazard_Model_v03_final.ipynb
rows = []
for e in EVENTS:
    pred = pd.DataFrame(index=tr.index[tr["Event_ID"] == e], columns=["FR", "RF", "LGBM"], dtype=float)
    for k in range(N_FOLDS):
        trn = ((tr["Fold"] != k) & (tr["Event_ID"] != e)).values
        tst = ((tr["Fold"] == k) & (tr["Event_ID"] == e)).values
        for name in ["FR", "RF", "LGBM"]:
            mdl = fit_model(name, tr.loc[trn, FEATURES], tr.loc[trn, "y"].values)
            pred.loc[tr.index[tst], name] = predict_model(name, mdl, tr.loc[tst, FEATURES])
    yy = tr.loc[pred.index, "y"].values
    for name in ["FR", "RF", "LGBM"]:
        r = scores(yy, pred[name].values)
        pooled = met[(met["Model"] == name) & (met["Event"] == e)].iloc[0]
        rows.append({"Held_Out_Event": e, "Model": name, "LOEO_ROC": r["ROC_AUC"], "Pooled_ROC": pooled["ROC_AUC"],
                     "LOEO_PR": r["PR_AUC"], "Pooled_PR": pooled["PR_AUC"], "Base_Rate": r["Base_Rate"]})
    print(f"{e} done")
loeo = pd.DataFrame(rows).round(4)
loeo.to_csv(OUT / "loeo_v03.csv", index=False)
print(loeo.to_string(index=False))

Amphan done
Remal done
Yaas done
Held_Out_Event Model  LOEO_ROC  Pooled_ROC  LOEO_PR  Pooled_PR  Base_Rate
        Amphan    FR    0.6216      0.6277   0.1219     0.1247     0.0791
        Amphan    RF    0.6199      0.6441   0.1189     0.1260     0.0791
        Amphan  LGBM    0.6223      0.6470   0.1231     0.1355     0.0791
         Remal    FR    0.7102      0.7382   0.4866     0.5171     0.2751
         Remal    RF    0.6709      0.7341   0.4279     0.5024     0.2751
         Remal  LGBM    0.6891      0.7438   0.4341     0.5129     0.2751
          Yaas    FR    0.7617      0.7613   0.1252     0.1264     0.0469
          Yaas    RF    0.7378      0.7359   0.1245     0.1227     0.0469
          Yaas  LGBM    0.7430      0.7446   0.1212     0.1267     0.0469


In [6]:
# ===== Cell 6: S3 canopy-gradient transfer test =====
# Notebook: P3_M_Hazard_Model_v03_final.ipynb
lo, hi = TREE_TRANSFER_TEST
grpA = (tr["Tree_Fraction"] < TREE_TRANSFER_TRAIN_MAX).values
grpB = ((tr["Tree_Fraction"] >= lo) & (tr["Tree_Fraction"] < hi)).values
pA = {n: pd.Series(np.nan, index=tr.index) for n in ["FR", "LGBM"]}
pB = {n: pd.Series(np.nan, index=tr.index) for n in ["FR", "LGBM"]}
for k in range(N_FOLDS):
    trn = (tr["Fold"] != k).values & grpA
    for name in ["FR", "LGBM"]:
        mdl = fit_model(name, tr.loc[trn, FEATURES], tr.loc[trn, "y"].values)
        for grp, store in [(grpA, pA), (grpB, pB)]:
            tst = (tr["Fold"] == k).values & grp
            store[name].loc[tst] = predict_model(name, mdl, tr.loc[tst, FEATURES])
rows = []
for gname, grp, store in [(f"Test Tree<{TREE_TRANSFER_TRAIN_MAX}", grpA, pA), (f"Test Tree {lo}-{hi}", grpB, pB)]:
    for name in ["FR", "LGBM"]:
        r = scores(tr.loc[grp, "y"].values, store[name][grp].values)
        r.update({"Group": gname, "Model": name})
        if name == "LGBM":
            r["Mean_Predicted"] = store[name][grp].mean()
        rows.append(r)
s3 = pd.DataFrame(rows)[["Group", "Model", "N", "Base_Rate", "ROC_AUC", "PR_AUC", "PR_Lift", "Mean_Predicted"]].round(4)
s3.to_csv(OUT / "s3_canopy_transfer_v03.csv", index=False)
print("Trained only on Tree_Fraction < 0.1 (spatial folds). Group B labels are partly suppressed by canopy,")
print("so read ROC (ranking) as the main transfer evidence; Mean_Predicted > Base_Rate in B is expected.")
print(s3.to_string(index=False))

Trained only on Tree_Fraction < 0.1 (spatial folds). Group B labels are partly suppressed by canopy,
so read ROC (ranking) as the main transfer evidence; Mean_Predicted > Base_Rate in B is expected.
            Group Model     N  Base_Rate  ROC_AUC  PR_AUC  PR_Lift  Mean_Predicted
    Test Tree<0.1    FR 85608     0.1800   0.6527  0.2788   1.5494             NaN
    Test Tree<0.1  LGBM 85608     0.1800   0.6495  0.2737   1.5207          0.1811
Test Tree 0.1-0.3    FR 63111     0.0709   0.6418  0.1158   1.6328             NaN
Test Tree 0.1-0.3  LGBM 63111     0.0709   0.6215  0.0969   1.3658          0.1032


In [7]:
# ===== Cell 7: Final fit on all observable rows, predict all cells, area of applicability =====
# Notebook: P3_M_Hazard_Model_v03_final.ipynb
final = {n: fit_model(n, tr[FEATURES], tr["y"].values) for n in MODELS}
OUT_NAMES = {"LGBM": "Hazard_Prob_LGBM", "RF": "Hazard_Prob_RF", "Logistic": "Hazard_Prob_Logistic",
             "FR": "Hazard_Score_FR", "ElevOnly": "Hazard_Score_ElevOnly"}
for n, col in OUT_NAMES.items():
    cells[col] = predict_model(n, final[n], cells[FEATURES])

imp = final["LGBM"].booster_.feature_importance(importance_type="gain").astype(float)
w = imp / imp.sum()
print("AOA weights (LGBM gain):", {f: round(float(v), 3) for f, v in zip(FEATURES, w)})
tc = tr.drop_duplicates("Grid_ID")[["Grid_ID", "Fold"] + FEATURES].reset_index(drop=True)
mu, sd = tc[FEATURES].mean(), tc[FEATURES].std()
scale = lambda X: ((X[FEATURES] - mu) / sd).values * w
Ztr, Zall = scale(tc), scale(cells)
samp = rng.choice(len(Ztr), size=min(3000, len(Ztr)), replace=False)
Dm = pairwise_distances(Ztr[samp])
dbar = Dm[np.triu_indices_from(Dm, 1)].mean()
di_cv = np.empty(len(Ztr))
for k in range(N_FOLDS):
    inn = tc["Fold"].values == k
    d, _ = NearestNeighbors(n_neighbors=1).fit(Ztr[~inn]).kneighbors(Ztr[inn])
    di_cv[inn] = d[:, 0] / dbar
q1, q3 = np.percentile(di_cv, [25, 75])
AOA_THR = q3 + 1.5 * (q3 - q1)
d, _ = NearestNeighbors(n_neighbors=1).fit(Ztr).kneighbors(Zall)
di = d[:, 0] / dbar
is_tr = cells["Grid_ID"].isin(tc["Grid_ID"]).values
di[is_tr] = pd.Series(di_cv, index=tc["Grid_ID"]).loc[cells.loc[is_tr, "Grid_ID"]].values
cells["AOA_DI"] = di
cells["Is_Within_AOA"] = (di <= AOA_THR).astype(int)
print(f"dbar = {dbar:.4f} | AOA threshold DI = {AOA_THR:.4f}")

pop = cells["Population_Count"]
aoa = cells.groupby("Unobservable_Reason").agg(
    N_Cells=("Grid_ID", "size"), Pct_Within_AOA=("Is_Within_AOA", "mean"),
    Population=("Population_Count", "sum"), Median_Elev=("Elevation_Mean_m", "median"),
    Median_Cropland=("Cropland_Fraction", "median"), Mean_Hazard_LGBM=("Hazard_Prob_LGBM", "mean"))
aoa["Pct_Within_AOA"] *= 100
print(aoa.round(3).to_string())
print(f"\nAll cells within AOA: {100 * cells['Is_Within_AOA'].mean():.2f}% | "
      f"population within AOA: {100 * pop[cells['Is_Within_AOA'] == 1].sum() / pop.sum():.2f}%")

AOA weights (LGBM gain): {'Elevation_Mean_m': 0.258, 'Slope_Mean_deg': 0.161, 'Distance_to_River_m': 0.169, 'Cropland_Fraction': 0.412}
dbar = 0.6174 | AOA threshold DI = 0.1137
                     N_Cells  Pct_Within_AOA   Population  Median_Elev  Median_Cropland  Mean_Hazard_LGBM
Unobservable_Reason                                                                                      
Mangrove               29017          11.269    15111.715        4.975            0.000             0.005
Observable             56174          85.944  1508427.373        1.906            0.872             0.129
Tree_And_Mangrove       1183          22.063    18006.265        2.955            0.000             0.034
Tree_Canopy            74500          46.097  6183570.401        2.375            0.292             0.028

All cells within AOA: 53.55% | population within AOA: 49.91%


In [8]:
# ===== Cell 8: Save, compare v02 vs final v03 =====
# Notebook: P3_M_Hazard_Model_v03_final.ipynb
out_cols = ["Grid_ID", "District"] + list(OUT_NAMES.values()) + ["Is_Observable", "Unobservable_Reason",
                                                                "AOA_DI", "Is_Within_AOA"]
assert len(cells) == N_MODEL and cells[list(OUT_NAMES.values())].notna().all().all()
cells[out_cols].to_csv(OUT / "hazard_predictions_v03.csv", index=False)
joblib.dump(final, OUT / "final_models_v03.joblib")
cfg = {"version": "v03_final", "date": "2026-10-29", "features": FEATURES, "target": TARGET,
       "label_file": P_LABEL.name, "mask_file": P_MASK.name, "weak_label_policy": "keep",
       "observability": "Tree_Fraction < 0.3 and Mangrove_Fraction_GMW <= 0.1",
       "cv": f"StratifiedGroupKFold {N_FOLDS} folds, {BLOCK_M} m blocks, seed {SEED}",
       "n_train_rows": int(len(tr)), "n_train_cells": int(tr["Grid_ID"].nunique()),
       "rf_params": RF_PARAMS, "lgbm_params": LGBM_PARAMS, "fr_bins": FR_BINS,
       "aoa": {"weights": {f: round(float(v), 4) for f, v in zip(FEATURES, w)}, "dbar": float(dbar),
               "threshold": float(AOA_THR), "rule": "Q3 + 1.5 IQR of CV dissimilarity index"},
       "n_boot": N_BOOT,
       "feature_tests": "Rel_Elev, Embankment, Waterway, HAND/TWI tested and not adopted (see f2/f3 files)"}
json.dump(cfg, open(OUT / "model_config_v03.json", "w"), indent=1)

h2 = pd.read_csv(P_HAZ_V02, usecols=["Grid_ID", "Hazard_Prob_LGBM"]).rename(columns={"Hazard_Prob_LGBM": "Hazard_v02"})
cmp_ = cells.merge(h2, on="Grid_ID", how="left", validate="1:1")
assert cmp_["Hazard_v02"].notna().all()
bins = [0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
pt = pd.DataFrame({
    "Pop_Pct_v02": cmp_.groupby(pd.cut(cmp_["Hazard_v02"], bins, include_lowest=True), observed=False)["Population_Count"].sum(),
    "Pop_Pct_v03": cmp_.groupby(pd.cut(cmp_["Hazard_Prob_LGBM"], bins, include_lowest=True), observed=False)["Population_Count"].sum()})
pt = (100 * pt / cmp_["Population_Count"].sum()).round(2)
pt.to_csv(OUT / "population_by_hazard_v02_vs_v03.csv")
print("Population share by hazard bin (%):\n", pt.to_string())
tb = cmp_.groupby(pd.cut(cmp_["Tree_Fraction"], [0, 0.1, 0.3, 0.5, 1.0], include_lowest=True), observed=False)[
    ["Hazard_v02", "Hazard_Prob_LGBM"]].mean().round(4)
print("\nMean hazard by Tree_Fraction bin:\n", tb.to_string())
print("\nSpearman v02 vs v03 (all cells):", round(cmp_["Hazard_v02"].corr(cmp_["Hazard_Prob_LGBM"], method="spearman"), 3))
print("\nSaved in:", OUT)
for f in sorted(OUT.iterdir()):
    print("  ", f.name)

Population share by hazard bin (%):
                 Pop_Pct_v02  Pop_Pct_v03
(-0.001, 0.01]        79.00        43.00
(0.01, 0.02]           3.35         9.89
(0.02, 0.05]           5.63        21.60
(0.05, 0.1]            4.92        16.41
(0.1, 0.2]             4.68         7.02
(0.2, 0.5]             2.34         2.05
(0.5, 1.0]             0.07         0.03

Mean hazard by Tree_Fraction bin:
                Hazard_v02  Hazard_Prob_LGBM
Tree_Fraction                              
(-0.001, 0.1]      0.0975            0.0919
(0.1, 0.3]         0.0669            0.0727
(0.3, 0.5]         0.0098            0.0419
(0.5, 1.0]         0.0000            0.0226

Spearman v02 vs v03 (all cells): 0.68

Saved in: D:\Siam\Thesis_GeoAI\05_models\final_v03
   ab_cell_level_target_v03.csv
   ab_relelev_bootstrap_v03.csv
   ab_relelev_metrics_v03.csv
   bootstrap_ci_v03.csv
   calibration_v03.csv
   f2_bootstrap_v03.csv
   f2_feature_metrics_v03.csv
   f3_hand_twi_bootstrap_v03.csv
   f3_hand_twi_m

In [9]:
# ===== Cell 9: Rank-based v02 vs v03 comparison (calibration-independent) =====
# Notebook: P3_M_Hazard_Model_v03_final.ipynb  (run after Cell 8, same kernel)
cmp_["Pct_v02"] = cmp_["Hazard_v02"].rank(pct=True, method="average")
cmp_["Pct_v03"] = cmp_["Hazard_Prob_LGBM"].rank(pct=True, method="average")
print("Mean predicted probability: v02 =", round(cmp_["Hazard_v02"].mean(), 4),
      "| v03 =", round(cmp_["Hazard_Prob_LGBM"].mean(), 4), "(different scales -> compare ranks)")

tb_bins = pd.cut(cmp_["Tree_Fraction"], [0, 0.1, 0.3, 0.5, 1.0], include_lowest=True)
g = cmp_.groupby(tb_bins, observed=False)
rank_tab = pd.DataFrame({
    "N_Cells": g.size(),
    "Population": g["Population_Count"].sum().round(0),
    "PopW_Mean_Pct_v02": g.apply(lambda d: np.average(d["Pct_v02"], weights=d["Population_Count"] + 1e-9)),
    "PopW_Mean_Pct_v03": g.apply(lambda d: np.average(d["Pct_v03"], weights=d["Population_Count"] + 1e-9)),
}).round(3)
rank_tab.to_csv(OUT / "rank_comparison_v02_vs_v03.csv")
print("\nPopulation-weighted mean hazard PERCENTILE by Tree_Fraction (0.5 = middle):\n", rank_tab.to_string())

for col, nm in [("Hazard_v02", "v02"), ("Hazard_Prob_LGBM", "v03")]:
    print(f"Spearman(hazard, Tree_Fraction) {nm}: {cmp_[col].corr(cmp_['Tree_Fraction'], method='spearman'):.3f}")

for nm, pc in [("v02", "Pct_v02"), ("v03", "Pct_v03")]:
    low = cmp_[pc] <= 0.2
    print(f"{nm}: population in the lowest-20% hazard cells (by rank): "
          f"{100 * cmp_.loc[low, 'Population_Count'].sum() / cmp_['Population_Count'].sum():.1f}%")

Mean predicted probability: v02 = 0.0486 | v03 = 0.0591 (different scales -> compare ranks)

Population-weighted mean hazard PERCENTILE by Tree_Fraction (0.5 = middle):
                N_Cells  Population  PopW_Mean_Pct_v02  PopW_Mean_Pct_v03
Tree_Fraction                                                           
(-0.001, 0.1]    62303    584466.0              0.834              0.769
(0.1, 0.3]       22888    939073.0              0.732              0.632
(0.3, 0.5]       21381   1268467.0              0.586              0.526
(0.5, 1.0]       54302   4933109.0              0.185              0.367
Spearman(hazard, Tree_Fraction) v02: -0.656
Spearman(hazard, Tree_Fraction) v03: -0.130
v02: population in the lowest-20% hazard cells (by rank): 37.3%
v03: population in the lowest-20% hazard cells (by rank): 16.0%
